In [ ]:
# ==========================================================
# STEP 1 : Import required libraries
# Purpose:
# Import all required LangChain, AWS Bedrock and Qdrant classes.
# ==========================================================

from langchain_community.document_loaders import TextLoader

from langchain.text_splitter import RecursiveCharacterTextSplitter

from langchain_aws import (
    BedrockEmbeddings,
    ChatBedrockConverse
)

from langchain_qdrant import QdrantVectorStore

from qdrant_client import QdrantClient

from langchain_core.prompts import ChatPromptTemplate

In [ ]:
# ==========================================================
# STEP 2 : Load document
#
# Purpose:
# Read the document that will become our knowledge base.
# In production this could be PDF, Word, SharePoint, S3 etc.
# ==========================================================

loader = TextLoader("sample.txt")

documents = loader.load()

In [ ]:
# ==========================================================
# STEP 3 : Chunk the document
#
# Purpose:
# LLMs cannot process very large documents.
# Split into small chunks before creating embeddings.
# ==========================================================

splitter = RecursiveCharacterTextSplitter(

    chunk_size=500,

    chunk_overlap=50

)

chunks = splitter.split_documents(documents)

In [ ]:
# ==========================================================
# STEP 4 : Create Embedding Model
#
# Purpose:
# Convert text into vectors using AWS Titan Embedding Model.
# ==========================================================

embeddings = BedrockEmbeddings(

    region_name="us-east-1",

    model_id="amazon.titan-embed-text-v2:0"

)

In [ ]:
# ==========================================================
# STEP 5 : Connect to Qdrant
#
# Purpose:
# Connect to local Qdrant database.
#
# Production:
# Qdrant Cloud
# ECS
# Kubernetes
# ==========================================================

client = QdrantClient(

    host="localhost",

    port=6333

)

In [ ]:
# ==========================================================
# STEP 6 : Store Embeddings in Qdrant
#
# Purpose:
# Convert every chunk into embeddings
# and store inside Vector Database.
# ==========================================================

vector_store = QdrantVectorStore.from_documents(

    documents=chunks,

    embedding=embeddings,

    client=client,

    collection_name="healthcare"

)

In [ ]:
# ==========================================================
# STEP 7 : Create Retriever
#
# Purpose:
# Retriever searches the Vector DB
# and returns only relevant chunks.
# ==========================================================

retriever = vector_store.as_retriever(

    search_kwargs={"k":3}

)

In [ ]:
# ==========================================================
# STEP 8 : Create Bedrock Chat Model
#
# Purpose:
# This LLM generates the final answer.
# ==========================================================

llm = ChatBedrockConverse(

    model="anthropic.claude-3-5-sonnet-20241022-v2:0",

    region_name="us-east-1"

)

In [ ]:
# ==========================================================
# STEP 9 : Prompt Template
#
# Purpose:
# Combine Question + Retrieved Context.
# ==========================================================

prompt = ChatPromptTemplate.from_template("""

Answer ONLY using the context.

Context:
{context}

Question:
{question}

""")

In [ ]:
# ==========================================================
# STEP 10 : Ask Question
#
# Purpose:
# User asks a question.
# ==========================================================

question = "What is diabetes?"

In [ ]:
# ==========================================================
# STEP 11 : Retrieve Documents
#
# Purpose:
# Search Qdrant for Top-K relevant chunks.
# ==========================================================

docs = retriever.invoke(question)

In [ ]:
# ==========================================================
# STEP 12 : Build Context
#
# Purpose:
# Convert retrieved documents into one string.
# ==========================================================

context = "\n\n".join(

    doc.page_content

    for doc in docs

)

In [ ]:
# ==========================================================
# STEP 13 : Create Final Prompt
#
# Purpose:
# Merge Question and Context.
# ==========================================================

messages = prompt.invoke(

    {

        "question": question,

        "context": context

    }

)

In [ ]:
# ==========================================================
# STEP 14 : Call Bedrock
#
# Purpose:
# Claude generates answer using retrieved context.
# ==========================================================

response = llm.invoke(messages)

print(response.content)


---

# Complete Flow

```text
                sample.txt
                     │
                     ▼
              TextLoader
                     │
                     ▼
      RecursiveCharacterTextSplitter
                     │
                     ▼
      AWS Titan Embeddings
                     │
                     ▼
              Qdrant Vector DB
                     │
──────────────────────────────────────
User Question
        │
        ▼
Retriever
        │
        ▼
Top 3 Chunks
        │
        ▼
ChatPromptTemplate
        │
        ▼
Claude (Bedrock)
        │
        ▼
Final Answer
```

---

# Interview Explanation

If the interviewer asks:

> **Explain this code.**

You can answer:

1. **Load Document** → Read the knowledge source.
2. **Chunking** → Split large documents into manageable pieces.
3. **Embedding Model** → Convert chunks into vectors using AWS Titan.
4. **Qdrant** → Store vectors for semantic search.
5. **Retriever** → Fetch the top 3 relevant chunks for the user's question.
6. **Prompt Template** → Combine retrieved context with the user's question.
7. **Bedrock (Claude)** → Generate the final answer based only on the retrieved context.

This is the **standard RAG pipeline** used in enterprise AI applications.

---

### Small Production Improvements

In a real enterprise application, you would additionally include:

- **FastAPI** as the API layer.
- **Amazon S3 / Azure Blob** for document storage instead of local files.
- **Amazon SQS / Azure Service Bus** for asynchronous document ingestion.
- **Redis** for caching frequent responses.
- **PostgreSQL** for chat history and metadata.
- **LangSmith** for tracing.
- **CloudWatch / Azure Monitor** for infrastructure monitoring.
- **JWT + RBAC** for authentication and authorization.

This keeps the core RAG logic simple while matching the architecture you've been studying.